# CashCompass: ML Training Notebook (Member 3 · Models 1 and 2)

This notebook trains **two small on-device models** and exports them for the Android app.

| # | Model | What it does | Output |
|---|---|---|---|
| 1 | **Monthly Financial Model** | Reads the last 6 months of *income, expenses, EMI* and predicts next month's **surplus** as a range | P10 / P50 / P90 surplus |
| 2 | **Risk Analysis Model** | Reads financial ratios + 5 questionnaire answers and rates the user's financial risk | **risk score (0-100)**, **risk category** (Low / Medium / High), **risk scope + flags**, and a readable **answer** |

**Project rules this notebook follows**
- No real user data. Data is **synthetic** (plus an optional public dataset for a sanity check).
- Fully reproducible: fixed seed, library versions and SHA-256 hashes are saved.
- Money is stored as whole rupees here. In the app it is integer paise; the model only sees **ratios**, so scale does not matter.
- The ML model *supports* the deterministic forecast engine; it never replaces the step-by-step maths shown to the user.

**Sections:** 0 Setup · 1 Data download · 2 Synthetic dataset · 3 Model 1 · 4 Model 2 · 5 Export · 6 Manifest

## 0. Setup

In [ ]:
%pip install -q numpy pandas scikit-learn matplotlib torch onnx

In [ ]:
import os, json, hashlib, random, platform, datetime
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import sklearn

SEED       = 42
N_USERS    = 3000      # synthetic people
N_MONTHS   = 24        # months of history per person
WINDOW     = 6         # months the models look back
EPOCHS     = 80
BATCH      = 256
OUT_DIR    = "outputs"      # exported models + reports
DATA_DIR   = "data"         # datasets
for d in (OUT_DIR, f"{DATA_DIR}/raw", f"{DATA_DIR}/processed"):
    os.makedirs(d, exist_ok=True)

def seed_all(seed=SEED):
    random.seed(seed); np.random.seed(seed)
seed_all()
print("python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__, "| sklearn", sklearn.__version__)

In [ ]:
import torch, torch.nn as nn
torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", DEVICE)

## 1. Data download (public dataset, optional)

We need a **real-world sanity check** for the risk model. This cell downloads the public *German Credit* dataset (OpenML `credit-g`, 1000 loan applicants, license: CC BY 4.0).
It is **not** used for training. It is only used to confirm that "loan burden" features really carry risk signal.

If you are offline the cell skips safely and everything else still runs.

In [ ]:
from sklearn.datasets import fetch_openml
credit = None
try:
    credit = fetch_openml("credit-g", version=1, as_frame=True, parser="auto").frame
    credit.to_csv(f"{DATA_DIR}/raw/german_credit.csv", index=False)
    print("Downloaded:", credit.shape, "-> data/raw/german_credit.csv")
    display(credit.head(3))
except Exception as e:
    print("Download skipped (offline or blocked):", repr(e)[:120])

In [ ]:
# Sanity check: does a loan-burden style feature predict bad credit in REAL data?
if credit is not None:
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import cross_val_score
    X = credit[["duration","credit_amount","installment_commitment","age"]].astype(float)
    X["monthly_burden"] = X["credit_amount"] / X["duration"]
    y = (credit["class"] == "bad").astype(int)
    auc = cross_val_score(LogisticRegression(max_iter=2000), (X - X.mean())/X.std(), y, cv=5, scoring="roc_auc").mean()
    print(f"Real-data sanity check, ROC-AUC with loan-burden features: {auc:.3f}  (0.5 = no signal)")

## 2. Synthetic monthly financial dataset

Each synthetic person has a **persona** (stable salary, variable income, or gig income), a rent level, a lifestyle level and 0-2 loans (EMIs).
Realistic effects included: yearly growth, **festival-season spending (Oct/Nov)**, random **shock expenses**, a **new loan starting mid-history**, and an **irregular bonus** for some salaried people (the "bonus last year, none this year" situation).

Columns: `user_id, month, income, expenses, emi, surplus, balance, income_type`  where `surplus = income - expenses - emi`.

In [ ]:
def generate_dataset(n_users=N_USERS, n_months=N_MONTHS, seed=SEED):
    rng = np.random.default_rng(seed)
    n, T = n_users, n_months

    income_type = rng.choice(["salaried", "variable", "gig"], size=n, p=[0.55, 0.30, 0.15])
    base_income = np.exp(rng.normal(np.log(55000), 0.55, n)).clip(15000, 400000)
    vol   = np.select([income_type == "salaried", income_type == "variable"], [0.03, 0.18], 0.30)
    growth = rng.normal(0.05, 0.03, n)                    # yearly raise
    rent  = base_income * rng.uniform(0.10, 0.35, n)
    life  = base_income * rng.uniform(0.20, 0.50, n)      # variable spending level
    n_loans = rng.choice([0, 1, 2], size=n, p=[0.35, 0.45, 0.20])
    emi0  = base_income * n_loans * rng.uniform(0.08, 0.28, n)
    new_loan_start = np.where(rng.random(n) < 0.10, rng.integers(8, T - 2, n), 10**6)
    new_emi = base_income * rng.uniform(0.08, 0.20, n)
    opening = base_income * rng.uniform(0.5, 4.0, n)

    start = pd.Timestamp("2023-01-01")
    months = pd.date_range(start, periods=T, freq="MS")
    moy = months.month.values                              # 1..12
    t = np.arange(T)

    season = np.where(np.isin(moy, [10, 11]), 1.15, 1.0)   # festival spend
    season = season * np.where(moy == 12, 1.08, 1.0)

    income = np.zeros((n, T)); expenses = np.zeros((n, T)); emi = np.zeros((n, T))
    for i in range(n):
        inc = base_income[i] * (1 + growth[i] * t / 12) * (1 + vol[i] * rng.standard_normal(T))
        if income_type[i] == "gig":
            inc = np.where(rng.random(T) < 0.08, inc * 0.4, inc)         # bad month
        if income_type[i] == "salaried" and rng.random() < 0.40:
            inc = inc + np.where((moy == 3) & (t < 12), base_income[i], 0)  # bonus in year 1 only
        exp_ = rent[i] + life[i] * season * (1 + 0.12 * rng.standard_normal(T))
        shock = rng.random(T) < 0.05
        exp_ = exp_ + shock * base_income[i] * rng.uniform(0.2, 0.8, T)   # medical/repair etc.
        e = emi0[i] + np.where(t >= new_loan_start[i], new_emi[i], 0)
        income[i], expenses[i], emi[i] = inc.clip(0), exp_.clip(0), e

    income, expenses, emi = np.round(income), np.round(expenses), np.round(emi)
    surplus = income - expenses - emi
    balance = opening[:, None] + np.cumsum(surplus, axis=1)

    df = pd.DataFrame({
        "user_id": np.repeat(np.arange(n), T),
        "month": np.tile(months.strftime("%Y-%m"), n),
        "income": income.ravel().astype(int),
        "expenses": expenses.ravel().astype(int),
        "emi": emi.ravel().astype(int),
        "surplus": surplus.ravel().astype(int),
        "balance": balance.ravel().astype(int),
        "income_type": np.repeat(income_type, T),
    })
    arrays = dict(income=income, expenses=expenses, emi=emi, surplus=surplus, balance=balance,
                  income_type=income_type, moy=moy)
    return df, arrays

df, A = generate_dataset()
df.to_csv(f"{DATA_DIR}/processed/monthly_financial_synthetic.csv", index=False)
print(df.shape); display(df.head(8)); display(df.describe().round(0))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
df.groupby("month")[["income","expenses","emi"]].mean().plot(ax=ax[0], title="Average per month (all users)")
df["surplus"].clip(-60000, 120000).hist(bins=60, ax=ax[1]); ax[1].set_title("Surplus distribution")
df.groupby("income_type")["surplus"].median().plot.bar(ax=ax[2], title="Median surplus by income type")
plt.tight_layout(); plt.show()

## 3. Model 1: Monthly Financial Model (next-month surplus range)

**Task:** given the last 6 months of income, expenses and EMI, plus the *already known* EMI for next month and the calendar month, predict next month's **surplus** as **P10 / P50 / P90**.

**Design choices**
- Everything is divided by the person's average income in the window, so the model works for any income level.
- **Quantile (pinball) loss** gives a *range* instead of one misleading number, matching the "uncertain income" requirement.
- Users are split into train/val/test (no person appears in two splits, so there is no leakage).
- Baseline to beat: "next surplus = average of the last 3 surpluses".

In [ ]:
def build_m1_features(income, expenses, emi, moy, window=WINDOW):
    """Returns X [N, 3*window + 3], y [N] (surplus ratio), scale [N] (avg income), user [N]."""
    n, T = income.shape
    Xs, ys, sc, us, tgt = [], [], [], [], []
    for tm in range(window, T):
        sl = slice(tm - window, tm)
        m = income[:, sl].mean(axis=1).clip(1)                     # scale = avg income in window
        f = np.concatenate([income[:, sl] / m[:, None],
                            expenses[:, sl] / m[:, None],
                            emi[:, sl] / m[:, None],
                            (emi[:, tm] / m)[:, None],             # next month's EMI is known
                            np.full((n, 1), np.sin(2*np.pi*moy[tm]/12)),
                            np.full((n, 1), np.cos(2*np.pi*moy[tm]/12))], axis=1)
        y = (income[:, tm] - expenses[:, tm] - emi[:, tm]) / m
        Xs.append(f); ys.append(y); sc.append(m); us.append(np.arange(n)); tgt.append(np.full(n, tm))
    return (np.vstack(Xs).astype("float32"), np.concatenate(ys).astype("float32"),
            np.concatenate(sc).astype("float32"), np.concatenate(us), np.concatenate(tgt))

X1, y1, sc1, u1, tm1 = build_m1_features(A["income"], A["expenses"], A["emi"], A["moy"])
rng = np.random.default_rng(SEED); perm = rng.permutation(N_USERS)
tr_u, va_u, te_u = perm[:int(.70*N_USERS)], perm[int(.70*N_USERS):int(.85*N_USERS)], perm[int(.85*N_USERS):]
mask = lambda users: np.isin(u1, users)
Xtr, ytr = X1[mask(tr_u)], y1[mask(tr_u)]
Xva, yva = X1[mask(va_u)], y1[mask(va_u)]
Xte, yte, sc_te = X1[mask(te_u)], y1[mask(te_u)], sc1[mask(te_u)]
print("features:", X1.shape[1], "| train/val/test rows:", len(Xtr), len(Xva), len(Xte))

In [ ]:
QUANTILES = [0.1, 0.5, 0.9]

class SurplusNet(nn.Module):
    def __init__(self, d_in, hidden=64, n_q=3):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_in, hidden), nn.ReLU(),
                                 nn.Linear(hidden, hidden), nn.ReLU(),
                                 nn.Linear(hidden, n_q))
    def forward(self, x):
        return torch.sort(self.net(x), dim=-1).values      # guarantees P10 <= P50 <= P90

def pinball(pred, target, qs=QUANTILES):
    q = torch.tensor(qs, device=pred.device).view(1, -1)
    e = target.view(-1, 1) - pred
    return torch.maximum(q * e, (q - 1) * e).mean()

def fit(model, loss_fn, tr, va, epochs=EPOCHS, lr=2e-3, patience=10):
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    Xt, yt = [torch.tensor(a) for a in tr]; Xv, yv = [torch.tensor(a) for a in va]
    best, best_state, bad, hist = 1e9, None, 0, []
    for ep in range(epochs):
        model.train(); idx = torch.randperm(len(Xt))
        for i in range(0, len(Xt), BATCH):
            b = idx[i:i+BATCH]
            opt.zero_grad(); loss = loss_fn(model(Xt[b]), yt[b]); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad(): vl = loss_fn(model(Xv), yv).item()
        hist.append(vl)
        if vl < best - 1e-5: best, bad, best_state = vl, 0, {k: v.clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience: break
    model.load_state_dict(best_state); model.eval()
    print(f"stopped at epoch {ep+1}, best val loss {best:.4f}")
    return hist

seed_all(); torch.manual_seed(SEED)
m1 = SurplusNet(X1.shape[1])
hist1 = fit(m1, pinball, (Xtr, ytr), (Xva, yva))
plt.plot(hist1); plt.title("Model 1 validation loss"); plt.xlabel("epoch"); plt.show()

In [ ]:
# ---- Evaluation on unseen users ----
with torch.no_grad(): pred = m1(torch.tensor(Xte)).numpy()
rupees = lambda r: r * sc_te                                      # ratio -> rupees
p10, p50, p90 = rupees(pred[:,0]), rupees(pred[:,1]), rupees(pred[:,2])
actual = rupees(yte)

# baseline: average surplus of last 3 months
inc, exp_, emi_ = Xte[:, 0:WINDOW], Xte[:, WINDOW:2*WINDOW], Xte[:, 2*WINDOW:3*WINDOW]
base = rupees((inc - exp_ - emi_)[:, -3:].mean(axis=1))

metrics_m1 = {
    "mae_p50_rupees":      float(np.abs(p50 - actual).mean()),
    "mae_baseline_rupees": float(np.abs(base - actual).mean()),
    "coverage_p10_p90":    float(((actual >= p10) & (actual <= p90)).mean()),   # ideal ~0.80
    "below_p10_rate":      float((actual < p10).mean()),                        # ideal ~0.10
    "above_p90_rate":      float((actual > p90).mean()),                        # ideal ~0.10
    "test_rows": int(len(actual)),
}
print(json.dumps(metrics_m1, indent=2))
assert metrics_m1["mae_p50_rupees"] < metrics_m1["mae_baseline_rupees"], "Model must beat the baseline (regression gate)"
print("PASS: beats the 3-month-average baseline")

In [ ]:
# One user's forecast, visualised
u = np.where(u1[mask(te_u)] == te_u[0])[0][:12]
plt.figure(figsize=(8,3.5))
plt.plot(actual[u], "k.-", label="actual surplus"); plt.plot(p50[u], "b-", label="P50")
plt.fill_between(range(len(u)), p10[u], p90[u], alpha=.25, label="P10-P90")
plt.legend(); plt.title("Next-month surplus forecast (one test user)"); plt.show()

## 4. Model 2: Risk Analysis Model

**Inputs (13 numbers):** 8 financial ratios from the last 6 months + 5 questionnaire answers.

| Input | Meaning |
|---|---|
| `emi_ratio` | EMI ÷ income |
| `expense_ratio` | expenses ÷ income |
| `savings_rate` | surplus ÷ income |
| `income_cv` | income variability (std ÷ mean) |
| `neg_month_frac` | share of months with negative surplus |
| `min_surplus_ratio` | worst month's surplus ÷ income |
| `emergency_months` | savings balance ÷ monthly expenses |
| `income_trend` | income slope per month ÷ income |
| `q_job` | Job stability (1 = very unstable … 5 = very stable) |
| `q_dependents` | People financially dependent (0-4) |
| `q_insurance` | Insurance cover (0 none, 1 basic, 2 good) |
| `q_horizon` | Goal horizon (1 = under 1 yr … 5 = over 10 yr) |
| `q_loss` | Comfort with a 10% loss (1 = not at all … 5 = very comfortable) |

**Outputs**
- `risk_score` (0-100) and `risk_category` (Low / Medium / High)
- `risk_flags` (multi-label) which map to a **risk scope**: *Debt, Cash-flow, Liquidity, Income*
- a plain-language **answer**, built by code from the real numbers (so no invented figures)

> **Honest note on labels:** there is no public dataset with these exact labels, so labels come from a documented rule formula plus noise. The model learns a fast, smooth version of the rules. Replace the label function with real expert/advisor labels when you have them.

In [ ]:
FLAGS = ["HIGH_EMI_BURDEN", "LOW_SAVINGS", "VOLATILE_INCOME", "OVERSPENDING", "LOW_EMERGENCY_FUND"]
SCOPE = {"HIGH_EMI_BURDEN": "Debt", "LOW_SAVINGS": "Liquidity", "VOLATILE_INCOME": "Income",
         "OVERSPENDING": "Cash-flow", "LOW_EMERGENCY_FUND": "Liquidity"}
CATEGORIES = ["Low", "Medium", "High"]
FEATURES2 = ["emi_ratio","expense_ratio","savings_rate","income_cv","neg_month_frac","min_surplus_ratio",
             "emergency_months","income_trend","q_job","q_dependents","q_insurance","q_horizon","q_loss"]

clip01 = lambda x: np.clip(x, 0, 1)

def make_questionnaire(rng, income_type, n):
    stab = np.select([income_type=="salaried", income_type=="variable"], [4, 3], 2)
    q_job = np.clip(np.round(stab + rng.normal(0, 0.8, n)), 1, 5)
    q_dep = rng.choice([0,1,2,3,4], n, p=[.25,.25,.25,.15,.10]).astype(float)
    q_ins = rng.choice([0,1,2], n, p=[.35,.40,.25]).astype(float)
    q_hor = rng.integers(1, 6, n).astype(float)
    q_los = rng.integers(1, 6, n).astype(float)
    return np.stack([q_job, q_dep, q_ins, q_hor, q_los], axis=1)

def ratio_features(inc, exp_, emi_, bal):
    """inc/exp_/emi_: [N, WINDOW] rupees, bal: [N] balance at end of window."""
    m  = inc.mean(axis=1).clip(1)
    sur = inc - exp_ - emi_
    tt = np.arange(inc.shape[1]); tt = tt - tt.mean()
    slope = (inc * tt).sum(axis=1) / (tt**2).sum()
    return np.stack([
        emi_.mean(axis=1) / m,
        exp_.mean(axis=1) / m,
        sur.mean(axis=1) / m,
        inc.std(axis=1) / m,
        (sur < 0).mean(axis=1),
        sur.min(axis=1) / m,
        np.clip(bal.clip(0) / exp_.mean(axis=1).clip(1), 0, 12),
        slope / m,
    ], axis=1)

def label_risk(F, Q, rng, noise=3.0):
    emi_r, exp_r, sav, cv, neg, minsur, ef, trend = F.T
    q_job, q_dep, q_ins = Q[:,0], Q[:,1], Q[:,2]
    q_term = ((5-q_job)/4 + q_dep/4 + (2-q_ins)/2) / 3
    score = 100 * (0.30*clip01(emi_r/0.5) + 0.20*clip01(cv/0.4) + 0.20*clip01((0.3-sav)/0.3)
                   + 0.10*neg + 0.10*clip01((3-ef)/3) + 0.10*q_term)
    score = np.clip(score + rng.normal(0, noise, len(score)), 0, 100)
    cat = np.digitize(score, [LOW_T, HIGH_T])                       # 0 Low, 1 Medium, 2 High
    flags = np.stack([emi_r > 0.40, sav < 0.10, cv > 0.25, (exp_r > 0.85) | (neg > 0.34), ef < 3], axis=1).astype("float32")
    return score.astype("float32"), cat.astype("int64"), flags

LOW_T, HIGH_T = 18.0, 35.0   # category thresholds (tuned to give a balanced split; documented in the data card)

def build_m2_dataset(A, seed=SEED):
    rng = np.random.default_rng(seed + 1)
    n, T = A["income"].shape
    Q_user = make_questionnaire(rng, A["income_type"], n)
    Fs, Qs, users = [], [], []
    for end in (6, 12, 18, 24):                                     # 4 snapshots per user
        sl = slice(end - WINDOW, end)
        F = ratio_features(A["income"][:, sl], A["expenses"][:, sl], A["emi"][:, sl], A["balance"][:, end-1])
        Q = Q_user + (rng.random(Q_user.shape) < 0.05) * rng.integers(-1, 2, Q_user.shape)   # answers drift slightly
        Q[:, 0] = np.clip(Q[:, 0], 1, 5); Q[:, 1] = np.clip(Q[:, 1], 0, 4); Q[:, 2] = np.clip(Q[:, 2], 0, 2)
        Q[:, 3:] = np.clip(Q[:, 3:], 1, 5)
        Fs.append(F); Qs.append(Q); users.append(np.arange(n))
    F, Q, users = np.vstack(Fs), np.vstack(Qs), np.concatenate(users)
    score, cat, flags = label_risk(F, Q, rng)
    X = np.concatenate([F, Q], axis=1).astype("float32")
    return X, score, cat, flags, users

X2, s2, c2, f2, u2 = build_m2_dataset(A)
print("rows:", X2.shape, "| category share:", dict(zip(CATEGORIES, np.round(np.bincount(c2, minlength=3)/len(c2), 2))))
print("flag rates:", dict(zip(FLAGS, f2.mean(axis=0).round(2))))
pd.DataFrame(X2, columns=FEATURES2).assign(risk_score=s2, risk_category=[CATEGORIES[i] for i in c2]).to_csv(
    f"{DATA_DIR}/processed/risk_dataset_synthetic.csv", index=False)

In [ ]:
# split by user (no leakage), then standardise using TRAIN statistics only
m2 = lambda users: np.isin(u2, users)
Xtr2, Xva2, Xte2 = X2[m2(tr_u)], X2[m2(va_u)], X2[m2(te_u)]
mu2, sd2 = Xtr2.mean(axis=0), Xtr2.std(axis=0) + 1e-6
norm = lambda x: ((x - mu2) / sd2).astype("float32")

def pack(mask_):
    return (norm(X2[mask_]), s2[mask_]/100.0, c2[mask_], f2[mask_])
D_tr, D_va, D_te = pack(m2(tr_u)), pack(m2(va_u)), pack(m2(te_u))

class RiskNet(nn.Module):
    def __init__(self, d_in, hidden=64):
        super().__init__()
        self.trunk = nn.Sequential(nn.Linear(d_in, hidden), nn.ReLU(), nn.Linear(hidden, hidden), nn.ReLU())
        self.score = nn.Linear(hidden, 1)
        self.cat   = nn.Linear(hidden, 3)
        self.flags = nn.Linear(hidden, len(FLAGS))
    def forward(self, x):
        h = self.trunk(x)
        return torch.sigmoid(self.score(h)).squeeze(-1) * 100, self.cat(h), self.flags(h)   # score, category logits, flag logits

mse, ce, bce = nn.MSELoss(), nn.CrossEntropyLoss(), nn.BCEWithLogitsLoss()
def risk_loss(out, tgt):
    score, logits, fl = out
    s, c, f = tgt
    return mse(score / 100, s) * 10 + ce(logits, c) + bce(fl, f)

def fit_multi(model, tr, va, epochs=EPOCHS, lr=2e-3, patience=10):
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    T_ = lambda d: (torch.tensor(d[0]), (torch.tensor(d[1]), torch.tensor(d[2]), torch.tensor(d[3])))
    (Xt, yt), (Xv, yv) = T_(tr), T_(va)
    best, best_state, bad, hist = 1e9, None, 0, []
    for ep in range(epochs):
        model.train(); idx = torch.randperm(len(Xt))
        for i in range(0, len(Xt), BATCH):
            b = idx[i:i+BATCH]
            opt.zero_grad(); loss = risk_loss(model(Xt[b]), tuple(t[b] for t in yt)); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad(): vl = risk_loss(model(Xv), yv).item()
        hist.append(vl)
        if vl < best - 1e-5: best, bad, best_state = vl, 0, {k: v.clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience: break
    model.load_state_dict(best_state); model.eval()
    print(f"stopped at epoch {ep+1}, best val loss {best:.4f}")
    return hist

seed_all(); torch.manual_seed(SEED)
risk = RiskNet(X2.shape[1])
hist2 = fit_multi(risk, D_tr, D_va)
plt.plot(hist2); plt.title("Model 2 validation loss"); plt.xlabel("epoch"); plt.show()

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
with torch.no_grad():
    sc_p, lg, fl = risk(torch.tensor(D_te[0]))
sc_p = sc_p.numpy(); cat_p = lg.argmax(1).numpy(); flag_p = (torch.sigmoid(fl).numpy() > 0.5).astype(int)
true_score, true_cat, true_flags = D_te[1]*100, D_te[2], D_te[3].astype(int)

metrics_m2 = {
    "score_mae_points": float(np.abs(sc_p - true_score).mean()),
    "category_accuracy": float(accuracy_score(true_cat, cat_p)),
    "category_macro_f1": float(f1_score(true_cat, cat_p, average="macro")),
    "flags_macro_f1": float(f1_score(true_flags, flag_p, average="macro", zero_division=0)),
    "flags_f1_each": {f: float(f1_score(true_flags[:, i], flag_p[:, i], zero_division=0)) for i, f in enumerate(FLAGS)},
    "test_rows": int(len(true_cat)),
}
print(json.dumps(metrics_m2, indent=2))
print("Confusion matrix (rows = true Low/Med/High):\n", confusion_matrix(true_cat, cat_p))

In [ ]:
# ---- Turn model output into a readable ANSWER (numbers come from code, not from a language model) ----
def explain_risk(raw_features: dict):
    x = np.array([[raw_features[k] for k in FEATURES2]], dtype="float32")
    with torch.no_grad():
        s, lg, fl = risk(torch.tensor(norm(x)))
    score = float(s[0]); cat = CATEGORIES[int(lg.argmax(1))]
    flags = [FLAGS[i] for i, p in enumerate(torch.sigmoid(fl)[0].numpy()) if p > 0.5]
    scope = sorted({SCOPE[f] for f in flags})
    why = {
      "HIGH_EMI_BURDEN":    f"EMIs take {raw_features['emi_ratio']*100:.0f}% of your income (healthy is below 40%).",
      "LOW_SAVINGS":        f"You keep only {raw_features['savings_rate']*100:.0f}% of income after spending and EMIs (aim for 10%+).",
      "VOLATILE_INCOME":    f"Your income swings by about {raw_features['income_cv']*100:.0f}% month to month.",
      "OVERSPENDING":       f"Expenses are {raw_features['expense_ratio']*100:.0f}% of income and {raw_features['neg_month_frac']*100:.0f}% of months ended negative.",
      "LOW_EMERGENCY_FUND": f"Savings cover only {raw_features['emergency_months']:.1f} months of expenses (aim for 3+).",
    }
    reasons = [why[f] for f in flags]
    answer = (f"Your financial risk is {cat.upper()} (score {score:.0f}/100). " +
              (("Main areas: " + ", ".join(scope) + ". " + " ".join(reasons)) if flags else "No major risk areas were found."))
    return {"risk_score": round(score, 1), "risk_category": cat, "risk_scope": scope, "risk_flags": flags, "answer": answer}

demo = dict(emi_ratio=0.46, expense_ratio=0.80, savings_rate=0.04, income_cv=0.30, neg_month_frac=0.33,
            min_surplus_ratio=-0.20, emergency_months=1.2, income_trend=0.0,
            q_job=2, q_dependents=2, q_insurance=0, q_horizon=3, q_loss=2)
print(json.dumps(explain_risk(demo), indent=2))

## 5. Export for the Android app

Saved to `outputs/`:

| File | Used by |
|---|---|
| `monthly_surplus_model.pt`, `risk_model.pt` | PyTorch checkpoints (re-training / debugging) |
| `*.onnx` | portable models, handy for quick tests |
| `*.pte` | **ExecuTorch** format for the app (only produced if `executorch` is installed) |
| `feature_spec.json` | **Contract for M1 (TypeScript):** exact feature order and normalisation. The TS code must compute inputs the same way. |
| `golden_fixtures.json` | Input/output pairs so M1 and M3 can verify their code matches this notebook |
| `EVAL_MONTHLY.json`, `EVAL_RISK.json` | Metrics for the CI eval gate (M4) |

> Models are exported in float32 here. Int4/Int8 quantisation (PT2E) is the next step in M2's pipeline for the final size budget; these models are tiny (tens of KB), so size is not a concern.

In [ ]:
m1.eval(); risk.eval()
torch.save(m1.state_dict(),   f"{OUT_DIR}/monthly_surplus_model.pt")
torch.save(risk.state_dict(), f"{OUT_DIR}/risk_model.pt")

ex1 = torch.tensor(Xte[:1]); ex2 = torch.tensor(D_te[0][:1])

# --- ONNX ---
try:
    torch.onnx.export(m1, ex1, f"{OUT_DIR}/monthly_surplus_model.onnx", input_names=["features"], output_names=["p10_p50_p90"], opset_version=17)
    torch.onnx.export(risk, ex2, f"{OUT_DIR}/risk_model.onnx", input_names=["features"], output_names=["score","category_logits","flag_logits"], opset_version=17)
    print("ONNX exported")
except Exception as e:
    print("ONNX export skipped:", repr(e)[:150])

# --- ExecuTorch (.pte) ---
try:
    from executorch.exir import to_edge
    from executorch.backends.xnnpack.partition.xnnpack_partitioner import XnnpackPartitioner
    for name, mdl, ex in (("monthly_surplus_model", m1, ex1), ("risk_model", risk, ex2)):
        prog = to_edge(torch.export.export(mdl, (ex,))).to_backend(XnnpackPartitioner()).to_executorch()
        with open(f"{OUT_DIR}/{name}.pte", "wb") as f: f.write(prog.buffer)
    print(".pte exported (XNNPACK)")
except Exception as e:
    print("ExecuTorch export skipped (install `executorch` to enable):", repr(e)[:120])

In [ ]:
feature_spec = {
  "version": 1,
  "window_months": WINDOW,
  "money_unit": "ratios only (divide every money value by the average income of the window)",
  "monthly_surplus_model": {
    "input_order": ([f"income_t-{WINDOW-i}/avg_income" for i in range(WINDOW)] +
                    [f"expenses_t-{WINDOW-i}/avg_income" for i in range(WINDOW)] +
                    [f"emi_t-{WINDOW-i}/avg_income" for i in range(WINDOW)] +
                    ["next_month_emi/avg_income", "sin(2*pi*month/12)", "cos(2*pi*month/12)"]),
    "output": ["surplus_p10_ratio", "surplus_p50_ratio", "surplus_p90_ratio"],
    "to_rupees": "multiply each output by avg_income of the window",
    "normalisation": "none (inputs are already ratios)",
  },
  "risk_model": {
    "input_order": FEATURES2,
    "normalisation": {"mean": mu2.tolist(), "std": sd2.tolist(), "formula": "(x - mean) / std"},
    "output": ["risk_score_0_100", "category_logits[Low,Medium,High]", f"flag_logits{FLAGS}"],
    "flag_threshold": 0.5, "flag_to_scope": SCOPE, "category_thresholds": {"low_below": LOW_T, "high_from": HIGH_T},
  },
}
json.dump(feature_spec, open(f"{OUT_DIR}/feature_spec.json", "w"), indent=2)

with torch.no_grad():
    g1 = m1(torch.tensor(Xte[:10])).numpy(); g2 = risk(torch.tensor(D_te[0][:10]))
golden = {
  "monthly_surplus_model": [{"input": Xte[i].tolist(), "output": g1[i].tolist()} for i in range(10)],
  "risk_model": [{"input_raw": X2[m2(te_u)][i].tolist(), "score": float(g2[0][i]),
                  "category": CATEGORIES[int(g2[1][i].argmax())]} for i in range(10)],
}
json.dump(golden, open(f"{OUT_DIR}/golden_fixtures.json", "w"), indent=2)

meta = {"seed": SEED, "date": datetime.date.today().isoformat(), "python": platform.python_version(),
        "torch": torch.__version__, "numpy": np.__version__, "sklearn": sklearn.__version__,
        "n_users": N_USERS, "n_months": N_MONTHS, "data": "synthetic (see data card)"}
json.dump({**meta, "metrics": metrics_m1}, open(f"{OUT_DIR}/EVAL_MONTHLY.json", "w"), indent=2)
json.dump({**meta, "metrics": metrics_m2}, open(f"{OUT_DIR}/EVAL_RISK.json", "w"), indent=2)
print(sorted(os.listdir(OUT_DIR)))

## 6. SHA-256 manifest and data card

The manifest lets M3's Gradle task (`copyModelAssets`) and M4's CI verify that the exact files were copied into the app.

In [ ]:
def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""): h.update(chunk)
    return h.hexdigest()

manifest = {fn: {"sha256": sha256(f"{OUT_DIR}/{fn}"), "bytes": os.path.getsize(f"{OUT_DIR}/{fn}")}
            for fn in sorted(os.listdir(OUT_DIR)) if fn.endswith((".pt", ".onnx", ".pte", ".json")) and fn != "manifest.json"}
json.dump(manifest, open(f"{OUT_DIR}/manifest.json", "w"), indent=2)
print(json.dumps(manifest, indent=2))

In [ ]:
data_card = f"""# DATA_CARD: Monthly Financial + Risk models

**Source:** fully synthetic, generated in this notebook (seed {SEED}, {N_USERS} people x {N_MONTHS} months).
**Real data used for training:** none. No personal or bank data.
**Public data (sanity check only, not trained on):** OpenML credit-g (German Credit), CC BY 4.0.
**PII scrubbing:** not needed (no real people); no names, merchants or account numbers exist in the data.

## Known limitations / bias notes
- Personas are simplified (salaried / variable / gig). Real incomes have more patterns (irregular bonuses, family transfers).
- Risk labels are produced by a documented rule formula plus noise, so the model mirrors that formula. It is NOT validated against real defaults.
- Thresholds (Low < {LOW_T}, High >= {HIGH_T}) are product choices, not medical or regulatory standards.
- Not financial advice. The app must show the step-by-step arithmetic from the deterministic engine next to any model output.

## Label formula (risk score)
30% EMI burden + 20% income volatility + 20% low savings + 10% negative months + 10% low emergency fund + 10% questionnaire (job, dependents, insurance).
"""
open(f"{OUT_DIR}/DATA_CARD.md", "w").write(data_card); print("DATA_CARD.md written")

## Next steps
1. Hand `outputs/` to M3 (Gradle copies + verifies hashes) and `feature_spec.json` / `golden_fixtures.json` to M1.
2. Replace the synthetic risk labels with advisor-reviewed labels when available.
3. Add Int8/Int4 PT2E quantisation, then re-check the golden fixtures stay within tolerance.
4. Add a **forecast track record**: after each real month, compare predicted P10-P90 with actual surplus.